In [1]:
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
import csv
import json
import math
import os, sys
import numpy as np
from numpy.typing import NDArray
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
from matplotlib.gridspec import GridSpec


In [2]:
AGES = [0, 3, 6, 12, 18, 24, 48, 72, 96, 120, 144, 168, 192, 216]
DISPLAY_AGES = [12, 48, 120, 216]
K_LEVELS = [5, 13, 25, 41, 61, 85, 113]
DISPLAY_K = [5, 13, 25, 41, 61, 85, 113]
LAYOUTS = ["age-adaptive", "adult-helmet"]
LOBES = ["frontal", "left temporal", "right temporal", "central", "parietal", "occipital"]

HEAD_RADIUS_MM = {0: 54.4, 3: 63.7, 6: 68.1, 12: 72.4, 18: 74.5, 24: 76.0, 48: 79.2, 72: 79.6, 96: 80.9, 120: 82.3, 144: 83.7, 168: 84.9, 192: 86.0, 216: 86.5}
CORTEX_OFFSET_MM = {0: 6.5, 3: 7.6, 6: 8.2, 12: 8.7, 18: 8.9, 24: 9.1, 48: 9.5, 72: 9.6, 96: 9.7, 120: 9.9, 144: 10.1, 168: 10.2, 192: 10.3, 216: 10.4}
DEFAULT_SOURCE_DEPTH_MM = 9.0
OPM_STANDOFF_MM = 4.0
RIGID_STANDOFF_MM = 5.0
BASELINE_CELL_DIAMETER_MM = 10.0
BASELINE_PROBE_SIDE_MM = 10.0
CELL_DIAMETERS_MM = [0, 5, 10, 15, 20, 25]
PROBE_SIDES_MM = [10, 20, 30, 40, 50]
NOISE_FT = 250.0
Q_AM = 10e-9
MU0_OVER_4PI = 1.0e-7
FT_PER_T = 1.0e15

# Lambert equal-area projection radii.  A single value is used for every K so
# the relative angular template is directly comparable across ages and layouts.
SOURCE_LAMBERT_RADIUS = 1.52
SENSOR_LAMBERT_RADIUS = 1.28
SOURCE_RING_COUNT = 6

@dataclass(frozen=True)
class Geometry:
    age: int
    scalp_radius_mm: float
    cortex_radius_mm: float


def geometry(age: int, adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> Geometry:
    scalp = HEAD_RADIUS_MM[int(age)]
    cortex_offset = CORTEX_OFFSET_MM[int(age)]
    real_source_depth_mm = adult_source_depth_mm * (scalp-cortex_offset)/(HEAD_RADIUS_MM[216]-CORTEX_OFFSET_MM[216])
    return Geometry(age=int(age), scalp_radius_mm=scalp, cortex_radius_mm=scalp - cortex_offset - real_source_depth_mm)

def normalize(v: NDArray[np.float64]) -> NDArray[np.float64]:
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, 1e-15)


def lambert_inverse(xy: NDArray[np.float64]) -> NDArray[np.float64]:
    xy = np.asarray(xy, dtype=float)
    x = xy[:, 0]
    y = xy[:, 1]
    rho2 = x * x + y * y
    z = 1.0 - 0.5 * rho2
    s = np.sqrt(np.maximum(0.0, 1.0 - 0.25 * rho2))
    dirs = np.column_stack([x * s, y * s, z])
    return normalize(dirs)


def lambert_xy(dirs: NDArray[np.float64]) -> NDArray[np.float64]:
    dirs = normalize(dirs)
    z = np.clip(dirs[:, 2], -1.0, 1.0)
    denom = np.sqrt(np.maximum(1e-12, 1.0 + z))
    return np.column_stack([np.sqrt(2.0) * dirs[:, 0] / denom, np.sqrt(2.0) * dirs[:, 1] / denom])


def ring_xy(max_radius: float, ring_count: int, points_per_first_ring: int = 4) -> NDArray[np.float64]:
    """Concentric ring grid with exact fourfold symmetry."""
    pts: list[tuple[float, float]] = [(0.0, 0.0)]
    for j in range(1, ring_count + 1):
        r = max_radius * j / ring_count
        n = points_per_first_ring * j
        for t in range(n):
            th = 2.0 * math.pi * t / n
            # Avoid tiny roundoff asymmetries on the symmetry axes.
            x = 0.0 if abs(math.cos(th)) < 1e-14 else r * math.cos(th)
            y = 0.0 if abs(math.sin(th)) < 1e-14 else r * math.sin(th)
            pts.append((float(x), float(y)))
    arr = np.asarray(pts, dtype=float)
    order = np.lexsort((arr[:, 0], arr[:, 1]))
    # Keep the center first for clarity in display and sensor construction.
    center = np.where(np.linalg.norm(arr, axis=1) < 1e-12)[0][0]
    rest = [i for i in order if i != center]
    return arr[[center] + rest]


def source_ring_xy() -> NDArray[np.float64]:
    # More source points than sensor points, with rings of 8,16,...,56 around a center.
    pts = [(0.0, 0.0)]
    for j in range(1, SOURCE_RING_COUNT + 1):
        r = SOURCE_LAMBERT_RADIUS * j / SOURCE_RING_COUNT
        n = 8 * j
        for t in range(n):
            th = 2.0 * math.pi * t / n
            x = 0.0 if abs(math.cos(th)) < 1e-14 else r * math.cos(th)
            y = 0.0 if abs(math.sin(th)) < 1e-14 else r * math.sin(th)
            pts.append((float(x), float(y)))
    return np.asarray(pts, dtype=float)


def classify_source_xy(x: float, y: float) -> str:
    # Coordinates are top-view Lambert projection: y>0 anterior/frontal, y<0 posterior.
    # The sectors partition one continuous whole-head grid rather than disconnected patches.
    r = math.hypot(x, y)
    if r < 0.26:
        return "central"
    if y >= 0.62:
        return "frontal"
    if y <= -0.88:
        return "occipital"
    if abs(x) <= 0.40 and -0.48 <= y <= 0.52:
        return "central"
    if x <= -0.56 and -0.62 <= y <= 0.48:
        return "left temporal"
    if x >= 0.56 and -0.62 <= y <= 0.48:
        return "right temporal"
    return "parietal"


@lru_cache(maxsize=None)
def source_grid() -> tuple[NDArray[np.float64], NDArray[np.float64], tuple[str, ...]]:
    xy = source_ring_xy()
    dirs = lambert_inverse(xy)
    labels = tuple(classify_source_xy(float(x), float(y)) for x, y in xy)
    counts = {l: labels.count(l) for l in LOBES}
    if any(v == 0 for v in counts.values()):
        raise RuntimeError(f"empty source sector: {counts}")
    return dirs, xy, labels


def ring_count_for_k(k: int) -> int:
    # K = 1 + 4(1+2+...+m) = 1 + 2m(m+1)
    if k not in K_LEVELS:
        raise ValueError(f"Unsupported high-symmetry channel count {k}")
    for m in range(1, 20):
        if 1 + 2 * m * (m + 1) == k:
            return m
    raise ValueError(k)


@lru_cache(maxsize=None)
def sensor_template_xy(k: int) -> NDArray[np.float64]:
    m = ring_count_for_k(k)
    return ring_xy(SENSOR_LAMBERT_RADIUS, m, points_per_first_ring=4)


@lru_cache(maxsize=None)
def sensor_template_dirs(k: int) -> NDArray[np.float64]:
    return lambert_inverse(sensor_template_xy(k))


def source_orientations(source_dirs: NDArray[np.float64]) -> NDArray[np.float64]:
    zaxis = np.array([0.0, 0.0, 1.0])
    xaxis = np.array([1.0, 0.0, 0.0])
    out = []
    for n in source_dirs:
        e1 = np.cross(zaxis, n)
        if np.linalg.norm(e1) < 0.08:
            e1 = np.cross(xaxis, n)
        e1 = e1 / np.linalg.norm(e1)
        e2 = np.cross(n, e1)
        e2 = e2 / np.linalg.norm(e2)
        # Mildly mixed tangential orientation to avoid a symmetry-induced zero in top-view maps.
        q = 0.84 * e1 + 0.16 * e2
        out.append(q / np.linalg.norm(q))
    return np.asarray(out, dtype=float)


def tangent_basis(n: NDArray[np.float64]) -> tuple[NDArray[np.float64], NDArray[np.float64]]:
    n = normalize(np.asarray(n, dtype=float).reshape(1, 3))[0]
    ref = np.array([0.0, 0.0, 1.0])
    if abs(float(np.dot(ref, n))) > 0.90:
        ref = np.array([0.0, 1.0, 0.0])
    e1 = ref - np.dot(ref, n) * n
    e1 = e1 / max(np.linalg.norm(e1), 1e-15)
    e2 = np.cross(n, e1)
    e2 = e2 / max(np.linalg.norm(e2), 1e-15)
    return e1, e2


def sensor_radius_mm(age: int, layout: str) -> float:
    if layout == "adult-helmet":
        return HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM
    return HEAD_RADIUS_MM[int(age)] + OPM_STANDOFF_MM


def valid_footprint(cell_diameter_mm: float, probe_side_mm: float) -> bool:
    # The active cell may be idealized as point-like (0 mm), but the packaged probe still has a finite base in the footprint scans.
    return probe_side_mm > 0 and probe_side_mm >= cell_diameter_mm


def aperture_offsets(direction: NDArray[np.float64], diameter_mm: float) -> NDArray[np.float64]:
    if diameter_mm <= 0:
        return np.zeros((1, 3), dtype=float)
    n = normalize(np.asarray(direction).reshape(1, 3))[0]
    e1, e2 = tangent_basis(n)
    a = 0.5 * diameter_mm * 1.0e-3
    offsets = [np.zeros(3)]
    for j in range(8):
        ph = 2.0 * math.pi * j / 8.0
        offsets.append(0.70 * a * (math.cos(ph) * e1 + math.sin(ph) * e2))
    return np.asarray(offsets, dtype=float)


def geodesic_distance_matrix_mm(dirs: NDArray[np.float64], radius_mm: float) -> NDArray[np.float64]:
    c = np.clip(dirs @ dirs.T, -1.0, 1.0)
    return radius_mm * np.arccos(c)


def is_mechanically_feasible(age: int, layout: str, k: int, probe_side_mm: float) -> bool:
    if probe_side_mm <= 0:
        return True
    dirs = sensor_template_dirs(k)
    D = geodesic_distance_matrix_mm(dirs, sensor_radius_mm(age, layout))
    if len(dirs) <= 1:
        return True
    dmin = float(np.min(D[np.triu_indices_from(D, k=1)]))
    return dmin + 1e-9 >= probe_side_mm


def feasible_count(age: int, layout: str, probe_side_mm: float) -> int:
    feasible = [k for k in K_LEVELS if is_mechanically_feasible(age, layout, k, probe_side_mm)]
    return max(feasible) if feasible else 0


def compute_leadfield(age: int, layout: str, sensor_dirs: NDArray[np.float64],
                      cell_diameter_mm: float = BASELINE_CELL_DIAMETER_MM,
                      adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> NDArray[np.float64]:
    sdirs, _, _ = source_grid()
    geom = geometry(age, adult_source_depth_mm)
    rs = sdirs * (geom.cortex_radius_mm * 1.0e-3)
    qvec = source_orientations(sdirs) * Q_AM
    rad = sensor_radius_mm(age, layout) * 1.0e-3
    out = np.empty((len(sensor_dirs), len(sdirs)), dtype=float)
    for j, n in enumerate(sensor_dirs):
        center = n * rad
        offs = aperture_offsets(n, cell_diameter_mm)
        accum = np.zeros(len(sdirs), dtype=float)
        for off in offs:
            rr = center + off - rs
            dist = np.linalg.norm(rr, axis=1)
            B = MU0_OVER_4PI * np.cross(qvec, rr) / np.maximum(dist[:, None] ** 3, 1e-30)
            accum += B @ n
        out[j, :] = (accum / len(offs)) * FT_PER_T
    return out


@lru_cache(maxsize=None)
def leadfield(age: int, layout: str, k: int, cell_diameter_mm: float = BASELINE_CELL_DIAMETER_MM,
              adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> NDArray[np.float64]:
    return compute_leadfield(age, layout, sensor_template_dirs(k), cell_diameter_mm, adult_source_depth_mm)


def source_positions_mm(age: int, adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> NDArray[np.float64]:
    sdirs, _, _ = source_grid()
    return sdirs * geometry(age, adult_source_depth_mm).cortex_radius_mm


def source_distance_matrix_mm(age: int, adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> NDArray[np.float64]:
    pos = source_positions_mm(age, adult_source_depth_mm)
    diff = pos[:, None, :] - pos[None, :, :]
    return np.linalg.norm(diff, axis=2)


def posterior_metrics(L: NDArray[np.float64], age: int, noise_ft: float = NOISE_FT,
                      adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM) -> dict[str, float]:
    D = source_distance_matrix_mm(age, adult_source_depth_mm)
    G = L.T @ L
    n2 = np.diag(G)
    delta2 = np.maximum(n2[:, None] + n2[None, :] - 2.0 * G, 0.0) / (2.0 * noise_ft * noise_ft)
    logits = -delta2
    logits -= np.max(logits, axis=1, keepdims=True)
    P = np.exp(logits)
    P /= np.maximum(np.sum(P, axis=1, keepdims=True), 1e-300)
    exp_err = np.sum(P * D, axis=1)
    mass20 = np.sum(P * (D <= 20.0), axis=1)
    mass30 = np.sum(P * (D <= 30.0), axis=1)
    # Credible radii of the matched-field posterior, evaluated on the discrete source dictionary.
    r50 = []
    r90 = []
    for i in range(P.shape[0]):
        order = np.argsort(D[i])
        cdf = np.cumsum(P[i, order])
        r50.append(float(D[i, order[min(np.searchsorted(cdf, 0.50), len(order)-1)]]))
        r90.append(float(D[i, order[min(np.searchsorted(cdf, 0.90), len(order)-1)]]))
    r50 = np.asarray(r50, dtype=float)
    r90 = np.asarray(r90, dtype=float)
    col_norm = np.sqrt(np.maximum(n2, 0.0))
    # Effective rank from the nonzero singular values, evaluated through the
    # smaller sensor-space Gram matrix to avoid backend-specific SVD stalls.
    gram_rows = (L / noise_ft) @ (L / noise_ft).T
    evals = np.linalg.eigvalsh(0.5 * (gram_rows + gram_rows.T))
    sv = np.sqrt(np.maximum(evals, 1e-18))
    sv = sv[sv > 1e-12]
    if sv.size == 0:
        erank = 0.0
    else:
        psv = sv / np.sum(sv)
        erank = float(np.exp(-np.sum(psv * np.log(psv))))
    X = L / np.maximum(col_norm[None, :], 1e-15)
    C = np.clip(np.abs(X.T @ X), 0.0, 1.0)
    np.fill_diagonal(C, -np.inf)
    nearest_corr = float(np.median(np.max(C, axis=1)))
    Sep = np.sqrt(np.maximum(n2[:, None] + n2[None, :] - 2.0 * G, 0.0)) / max(noise_ft, 1e-15)
    np.fill_diagonal(Sep, np.inf)
    nearest_sep = float(np.median(np.min(Sep, axis=1)))
    return {
        "median_error": float(np.median(exp_err)),
        "p90_error": float(np.percentile(exp_err, 90)),
        "mean_error": float(np.mean(exp_err)),
        "success20": float(100.0 * np.mean(mass20)),
        "success30": float(100.0 * np.mean(mass30)),
        "median_r50": float(np.median(r50)),
        "median_r90": float(np.median(r90)),
        "snr_median": float(np.median(col_norm / noise_ft)),
        "effective_rank": erank,
        "nearest_corr": nearest_corr,
        "nearest_sep": nearest_sep,
        "resolution_length": float(np.median(exp_err)),
    }


def evaluate(age: int, layout: str, k: int,
             cell_diameter_mm: float = BASELINE_CELL_DIAMETER_MM,
             probe_side_mm: float = BASELINE_PROBE_SIDE_MM,
             adult_source_depth_mm: float = DEFAULT_SOURCE_DEPTH_MM,
             noise_ft: float = NOISE_FT) -> dict[str, float | int | str]:
    valid = valid_footprint(cell_diameter_mm, probe_side_mm) and is_mechanically_feasible(age, layout, k, probe_side_mm)
    scalp = HEAD_RADIUS_MM[int(age)]
    cortex_offset = CORTEX_OFFSET_MM[int(age)]
    real_source_depth_mm = adult_source_depth_mm * (scalp-cortex_offset)/(HEAD_RADIUS_MM[216]-CORTEX_OFFSET_MM[216])
    base = {"age": age, "layout": layout, "K": k, "selected": k if valid else 0, "valid": int(valid),
            "cell_mm": cell_diameter_mm, "probe_mm": probe_side_mm, "noise_ft": noise_ft, "depth_mm": real_source_depth_mm}
    if not valid:
        return {**base, "median_error": math.nan, "p90_error": math.nan, "mean_error": math.nan,
                "success20": math.nan, "success30": math.nan, "median_r50": math.nan, "median_r90": math.nan,
                "snr_median": math.nan, "effective_rank": math.nan, "nearest_corr": math.nan,
                "nearest_sep": math.nan, "resolution_length": math.nan}
    L = leadfield(age, layout, k, cell_diameter_mm, adult_source_depth_mm)
    return {**base, **posterior_metrics(L, age, noise_ft=noise_ft, adult_source_depth_mm=adult_source_depth_mm)}


def field_profile_from_kernel(age: int, theta: NDArray[np.float64], layout: str,
                              cell_diameter_mm: float = BASELINE_CELL_DIAMETER_MM) -> NDArray[np.float64]:
    geom = geometry(age)
    rs = np.array([0.0, 0.0, geom.cortex_radius_mm * 1e-3])
    q = np.array([0.0, Q_AM, 0.0])
    rad = sensor_radius_mm(age, layout) * 1e-3
    vals = []
    for th in theta:
        az = math.radians(38.0)
        n = np.array([math.sin(th) * math.cos(az), math.sin(th) * math.sin(az), math.cos(th)])
        offs = aperture_offsets(n, cell_diameter_mm)
        accum = 0.0
        for off in offs:
            rr = n * rad + off - rs
            dist = np.linalg.norm(rr)
            B = MU0_OVER_4PI * np.cross(q, rr) / max(dist ** 3, 1e-30)
            accum += abs(float(np.dot(B, n)))
        vals.append(accum / len(offs) * FT_PER_T)
    return np.asarray(vals, dtype=float)


def lobe_metrics(age: int, layout: str, k: int) -> list[dict]:
    L = leadfield(age, layout, k)
    D = source_distance_matrix_mm(age)
    _, _, labels = source_grid()
    labels_arr = np.asarray(labels, dtype=object)
    G = L.T @ L
    n2 = np.diag(G)
    delta2 = np.maximum(n2[:, None] + n2[None, :] - 2.0 * G, 0.0) / (2.0 * NOISE_FT * NOISE_FT)
    logits = -delta2; logits -= np.max(logits, axis=1, keepdims=True)
    P = np.exp(logits); P /= np.sum(P, axis=1, keepdims=True)
    exp_err = np.sum(P * D, axis=1)
    mass20 = np.sum(P * (D <= 20.0), axis=1)
    out = []
    for lobe in LOBES:
        js = np.where(labels_arr == lobe)[0]
        out.append({"age": age, "layout": layout, "K": k, "lobe": lobe,
                    "median_error": float(np.median(exp_err[js])),
                    "success20": 100.0 * float(np.mean(mass20[js]))})
    return out


def cross_age_matrix(k: int = 61) -> list[dict]:
    # With the fixed high-symmetry angular template there is no age-specific angular selection.
    # The matrix quantifies how the same K-template behaves across source ages and shell radii.
    rows = []
    for shell_age in AGES:
        for eval_age in AGES:
            # Use the age-adaptive radius of shell_age with sources at eval_age by explicitly computing geometry.
            # This diagnostic is most meaningful near the diagonal and is used only for consistency checks.
            L = leadfield(eval_age, "age-adaptive", k)
            m = posterior_metrics(L, eval_age)
            rows.append({"design_age": shell_age, "eval_age": eval_age, "K": k, **m})
    return rows


def random_control(ages: tuple[int, ...] = tuple(DISPLAY_AGES), k: int = 61, nrep: int = 30) -> list[dict]:
    # Random control: random rotations of ring phases that preserve ring counts but break the prescribed D4 phase relation.
    rows = []
    m = ring_count_for_k(k)
    for age in ages:
        rng = np.random.default_rng(20260721 + int(age) + k)
        for rep in range(nrep):
            pts = [(0.0, 0.0)]
            for j in range(1, m + 1):
                r = SENSOR_LAMBERT_RADIUS * j / m
                n = 4 * j
                phi = rng.uniform(0, 2.0 * math.pi / n)
                for t in range(n):
                    th = phi + 2.0 * math.pi * t / n
                    pts.append((r * math.cos(th), r * math.sin(th)))
            dirs = lambert_inverse(np.asarray(pts, dtype=float))
            L = compute_leadfield(int(age), "age-adaptive", dirs, BASELINE_CELL_DIAMETER_MM, DEFAULT_SOURCE_DEPTH_MM)
            mm = posterior_metrics(L, int(age))
            rows.append({"rep": rep, "age": int(age), "K": k, **mm})
    return rows


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    keys = list(rows[0].keys())
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader(); w.writerows(rows)


In [3]:
def load(out_dir: Path) -> dict:
    with (out_dir / "results.json").open() as f:
        return json.load(f)

base_dir = os.path.dirname(os.path.realpath('file'))
ROOT = Path(base_dir).resolve().parents[0]
out_dir = ROOT / "outputs"
fig_dir = ROOT / "figures"
res = load(out_dir)

In [4]:
out_dir.mkdir(parents=True, exist_ok=True)
sdirs, sxy, labels = source_grid()

summary = []
for age in AGES:
    for layout in LAYOUTS:
        for k in K_LEVELS:
            summary.append(evaluate(age, layout, k))
write_csv(out_dir / "summary.csv", summary)

noise_rows = []
for noise in np.arange(20,401,20):
    for layout in LAYOUTS:
        for age in DISPLAY_AGES:
            noise_rows.append(evaluate(age, layout, 61, noise_ft=float(noise)))
write_csv(out_dir / "noise.csv", noise_rows)

depth_rows = []
for depth in np.arange(1,17,1):
    for layout in LAYOUTS:
        for age in DISPLAY_AGES:
            depth_rows.append(evaluate(age, layout, 61, adult_source_depth_mm = float(depth)))
write_csv(out_dir / "depth.csv", depth_rows)

footprint = []
for cell in CELL_DIAMETERS_MM:
    compatible = [p for p in PROBE_SIDES_MM if p >= cell]
    if not compatible:
        continue
    probe = min(compatible)
    for age in AGES:
        for layout in LAYOUTS:
            footprint.append(evaluate(age, layout, 41, cell, probe))
write_csv(out_dir / "footprint.csv", footprint)

feas = []
for side in PROBE_SIDES_MM:
    for age in AGES:
        for layout in LAYOUTS:
            feas.append({"age": age, "layout": layout, "probe_mm": side, "feasible": feasible_count(age, layout, side)})
write_csv(out_dir / "feasibility.csv", feas)

lobes = []
for age in DISPLAY_AGES:
    for layout in LAYOUTS:
        lobes.extend(lobe_metrics(age, layout, 61))
write_csv(out_dir / "lobe.csv", lobes)

rand = random_control(ages=tuple(DISPLAY_AGES), k=61, nrep=30)
write_csv(out_dir / "random_control.csv", rand)

source_counts = [{"lobe": l, "count": labels.count(l)} for l in LOBES]
write_csv(out_dir / "source_counts.csv", source_counts)

results = {
    "source_dirs": sdirs.tolist(), "source_xy": sxy.tolist(), "source_labels": list(labels),
    "summary": summary, "noise": noise_rows, "depth": depth_rows, "footprint": footprint,
    "feasibility": feas, "lobe": lobes, "random_control": rand,
    "source_counts": source_counts,
    "constants": {"noise_ft": NOISE_FT, "baseline_cell_mm": BASELINE_CELL_DIAMETER_MM,
                  "baseline_probe_mm": BASELINE_PROBE_SIDE_MM, "q_Am": Q_AM,
                  "sensor_lambert_radius": SENSOR_LAMBERT_RADIUS,
                  "source_lambert_radius": SOURCE_LAMBERT_RADIUS}
}
with (out_dir / "results.json").open("w") as f:
    json.dump(results, f, indent=2)

In [5]:
# Low-saturation palette inspired by the fifth-series RMB notes, tuned for print.
RMB = {
    "red": "#9E2F3F",
    "blue": "#2F5E8F",
    "green": "#3F7D5A",
    "brown": "#8B6B3E",
    "purple": "#6E5A8A",
    "gold": "#B58B35",
    "gray": "#5E6670",
    "lightgray": "#C9CED4",
}
ORDERED = [RMB["red"], RMB["blue"], RMB["green"], RMB["brown"], RMB["purple"], RMB["gold"], RMB["gray"]]


def cm_to_inch(w_cm: float, h_cm: float) -> tuple[float, float]:
    return (w_cm / 2.54, h_cm / 2.54)


def set_style() -> None:
    mpl.rcParams.update({
        "font.family": ["Arial", "DejaVu Sans", "Times New Roman"],
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.25,
        "patch.linewidth": 0.8,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#444444",
        "grid.color": "#D9D9D9",
        "grid.linewidth": 0.4,
        "grid.alpha": 0.65,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def boxed_legend(ax, *args, **kwargs):
    defaults = dict(frameon=True, facecolor="white", edgecolor="#666666", framealpha=0.50,
                    borderpad=0.30, handlelength=1.8, handletextpad=0.45, columnspacing=0.75)
    defaults.update(kwargs)
    return ax.legend(*args, **defaults)


def panel_label(ax, label: str, x: float = -0.16, y: float = 1.08) -> None:
    ax.text(x, y, f"({label})", transform=ax.transAxes, ha="left", va="bottom",
            fontsize=9, fontweight="bold", clip_on=False, color="#202020")

In [6]:
MS = 2
SCATTER_S = 12.5
DISPLAY_AGE_COLORS = {12:"#005A8D", 48: "#C48A00", 120: "#178B5B", 216: "#8A4FA3"}
LAYOUT_COLORS = {"age-adaptive": "#D55F6F", "AA": "#D55F6F", 'scalp': "#D55F6F",
                "adult-helmet": "#6B6773",  "AH": "#6B6773", 'helmet': "#6B6773", 
                 'source': "#00A6B2",'sensor': "#1c3c31", 
                }

LOBE_COLORS = {
    "frontal": ORDERED[0], "left temporal": ORDERED[1], "right temporal": ORDERED[2],
    "central": ORDERED[5], "parietal": ORDERED[4], "occipital": ORDERED[3]
}
SHORT_LOBE = {"frontal":"front.", "left temporal":"L temp.", "right temporal":"R temp.", "central":"cent.", "parietal":"par.", "occipital":"occ."}


def line_style(layout: str) -> str:
    return "-" if layout == "age-adaptive" else "--"


def marker_style(layout: str) -> str:
    return "o" if layout == "age-adaptive" else "s"


def layout_label(layout: str) -> str:
    return "AA" if layout == "age-adaptive" else "AH"


def savefig(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    plt.close(fig)


def load(out_dir: Path) -> dict:
    with (out_dir / "results.json").open() as f:
        return json.load(f)




In [7]:
def cap_scaffold(ax, radius: float = 1.55, midline: bool = True, cross: bool = False) -> None:
    ax.add_patch(Circle((0, 0), radius, fill=False, ec="#A0A6AC", lw=0.75, ls="-"))
    if midline:
        ax.plot([0, 0], [-radius, radius], color="#878C92", lw=0.70, ls=":", zorder=0)
    if cross:
        ax.plot([-radius, radius], [0, 0], color="#B3B7BD", lw=0.55, ls="--", zorder=0)
    ax.set_aspect("equal")
    ax.set_xlim(-radius * 1.08, radius * 1.08)
    ax.set_ylim(-radius * 1.08, radius * 1.08)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


def semicircle(ax, cx, cy, r, color, lw=1.0, ls="-", alpha=1.0):
    th = np.linspace(0, math.pi, 220)
    ax.plot(cx + r * np.cos(th), cy + r * np.sin(th), color=color, lw=lw, ls=ls, alpha=alpha)

In [8]:
# Figure 1

set_style()
fig = plt.figure(figsize=cm_to_inch(18, 20.6))
gs = GridSpec(4, 6, figure=fig, height_ratios=[1.0, 0.82, 0.58, 0.58], hspace=0.42, wspace=0.70)

ax = fig.add_subplot(gs[0, 0:2])
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    scalp = HEAD_RADIUS_MM[age]
    cortex = geometry(age).cortex_radius_mm
    ax.add_patch(Circle((0, 0), scalp, fill=False, ec=col, lw=1.05, ls="-"))
    ax.add_patch(Circle((0, 0), cortex, fill=False, ec=col, lw=0.88, ls=":", alpha=0.86))
ax.add_patch(Circle((0, 0), HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM, fill=False, ec=LAYOUT_COLORS['helmet'], lw=0.92, ls="--"))
ax.set_aspect("equal")
lim = HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM + 8
ax.set_xlim(-105, 105); ax.set_ylim(-105, 105)
ax.set_xlabel("lateral position (mm)"); ax.set_ylabel("superior position (mm)")
ax.grid(True)
handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=1.2, label=f"{a} m") for a in DISPLAY_AGES]
handles += [Line2D([0], [0], color="#333333", lw=0.9, ls=":", label="source"),
            Line2D([0], [0], color="#333333", lw=1.0, ls="-", label="scalp"),
            Line2D([0], [0], color="#333333", lw=0.9, ls="--", label="helmet")]
boxed_legend(ax, handles=handles, loc="lower left", ncol=2, handlelength=1.10, columnspacing=0.45, framealpha=0.90)
panel_label(ax, "a")

ax = fig.add_subplot(gs[0, 2:4])
ax.plot(AGES, [geometry(a).cortex_radius_mm for a in AGES], marker="o", ms=MS, color=LAYOUT_COLORS['source'], ls=":", lw=0.88, label="source")
ax.plot(AGES, [HEAD_RADIUS_MM[a] for a in AGES], marker="o", ms=MS, color=LAYOUT_COLORS['scalp'], lw=1.05, label="scalp")
ax.plot(AGES, [HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM for _ in AGES], marker="o", ms=MS, color=LAYOUT_COLORS['helmet'], ls="--", lw=0.92, label="helmet")
ax.set_xlabel("age (m)"); ax.set_ylabel("radius (mm)"); ax.grid(True)
ax.set_ylim(30, 105)
boxed_legend(ax, loc="lower right", ncol=1, handlelength=2.6)
panel_label(ax, "b")

ax = fig.add_subplot(gs[0, 4:6])
xy = np.asarray(res["source_xy"]); labels = np.asarray(res["source_labels"], dtype=object)
cap_scaffold(ax, radius=SOURCE_LAMBERT_RADIUS, midline=True)
for lab in LOBES:
    m = labels == lab
    ax.scatter(xy[m, 0], xy[m, 1], s=SCATTER_S, color=LOBE_COLORS[lab], alpha=0.82,
               edgecolors="white", linewidths=0.20, label=SHORT_LOBE[lab])
boxed_legend(ax, loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.08), borderaxespad=0.0,
             handlelength=0.6, columnspacing=0.55, borderpad=0.22)
panel_label(ax, "c", x=-0.12)

ax = fig.add_subplot(gs[1, 0:3])
theta = np.linspace(0.04, 1.55, 180)
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    ax.plot(theta, field_profile_from_kernel(age, theta, "age-adaptive"), color=col, ls="-", lw=1.05)
    ax.plot(theta, field_profile_from_kernel(age, theta, "adult-helmet"), color=col, ls="--", lw=0.88)
ax.set_xlabel("angular separation (rad)"); ax.set_ylabel("normal field amplitude (fT)"); ax.grid(True)
age_handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=3, ls="-", label=f"{a} m") for a in DISPLAY_AGES]
style_handles = [Line2D([0], [0], color="#333333", lw=1.0, ls="-", label="AA"),
                 Line2D([0], [0], color="#333333", lw=0.9, ls="--", label="AH")]
boxed_legend(ax, handles=age_handles + style_handles, loc="upper right", ncol=3, borderpad=0.50, columnspacing=0.55, handlelength=1.88)
panel_label(ax, "d")



ax = fig.add_subplot(gs[1, 3:6])
ax.set_axis_off()
ax.set_xlim(0, 1); ax.set_ylim(0, 0.6)
cx1, cx2 = 0.24, 0.76
cy = 0.15
age = 12
scale = 0.235 / (HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM)
scalp = HEAD_RADIUS_MM[age] * scale
source = geometry(age).cortex_radius_mm * scale
opm_shell = (HEAD_RADIUS_MM[age] + OPM_STANDOFF_MM) * scale
helmet = (HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM) * scale
# Age-adaptive panel: all relevant surfaces share a center and are hemispherical arcs.
for cx, is_opm in [(cx1, True), (cx2, False)]:
    semicircle(ax, cx, cy, source, LAYOUT_COLORS['source'], lw=0.85, ls=":", alpha=0.92)
    semicircle(ax, cx, cy, scalp, LAYOUT_COLORS['scalp'], lw=1.05, ls="-", alpha=0.92)
    semicircle(ax, cx, cy, helmet, LAYOUT_COLORS['helmet'], lw=0.90, ls="--", alpha=0.95)
    meas = opm_shell if is_opm else helmet
    # Place a sparse angular template on the actual measurement hemisphere.
    ang = np.linspace(0.18 * math.pi, 0.82 * math.pi, 5)
    ax.scatter(cx + meas * np.cos(ang), cy + meas * np.sin(ang), s=10, zorder=4,
               color='k', edgecolors="white", linewidths=0.25)
ax.text(cx1, 0.07, "AA shell", ha="center", va="bottom", fontsize=7.2, color=LAYOUT_COLORS["AA"])
ax.text(cx2, 0.07, "AH shell", ha="center", va="bottom", fontsize=7.2, color=LAYOUT_COLORS["AH"])
ax.text(cx1, 0.01, "(radii scale with age)", ha="center", va="bottom", fontsize=6.4, color="#333333")
ax.text(cx2, 0.01, "(helmet radius fixed)", ha="center", va="bottom", fontsize=6.4, color="#333333")
handles = [
    Line2D([0], [0], color=LAYOUT_COLORS['source'], lw=0.88, ls=":", label="source"),
    Line2D([0], [0], color=LAYOUT_COLORS['scalp'], lw=1.05, label="scalp"),
    Line2D([0], [0], color=LAYOUT_COLORS['helmet'], lw=0.92, ls="--", label="helmet"),
    Line2D([0], [0], color=LAYOUT_COLORS['sensor'], lw=0.0, marker="o", ms=2, label="sensor"),
]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.50, 0.90), ncol=4,
          frameon=True, facecolor="white", edgecolor="#666666", framealpha=0.50,
          borderpad=0.30, handlelength=1.2, columnspacing=0.55, fontsize=6.4)
panel_label(ax, "e", x=-0.02, y=1.05)





sub = gs[2:4, 0:6].subgridspec(2, 4, hspace=0.04, wspace=0.10)
axes_f = []
for r, layout in enumerate(["age-adaptive", "adult-helmet"]):
    for c, age in enumerate(DISPLAY_AGES):
        ax = fig.add_subplot(sub[r, c]); axes_f.append(ax)
        col = DISPLAY_AGE_COLORS[age]
        scalp = HEAD_RADIUS_MM[age]
        source_r = geometry(age).cortex_radius_mm
        helmet = HEAD_RADIUS_MM[216] + RIGID_STANDOFF_MM
        sc = 1.0 / helmet
        lim = 1.08
        ax.add_patch(Circle((0, 0), helmet * sc, fill=False, ec=LAYOUT_COLORS['helmet'], lw=0.88, ls="--"))
        ax.add_patch(Circle((0, 0), scalp * sc, fill=False, ec=col, lw=1.05, ls="-"))
        ax.add_patch(Circle((0, 0), source_r * sc, fill=False, ec=col, lw=0.92, ls=":", alpha=0.90))
        ax.plot([0, 0], [-lim, lim], color="#8B8F94", lw=0.55, ls=":", zorder=0)
        base_xy = sensor_template_xy(41) / SENSOR_LAMBERT_RADIUS
        radius = scalp * sc if layout == "age-adaptive" else helmet * sc
        pts = base_xy * radius
        ax.scatter(pts[:, 0], pts[:, 1], s=5, color='k',
                   edgecolors="white", linewidths=0.18, zorder=4)
        ax.set_aspect("equal"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.text(0.02, 0.96, f"{age} m", transform=ax.transAxes, va="top", ha="left", fontsize=7.0, color=col)
        if c == 0:
            ax.text(-0.10, 0.50, layout_label(layout),
                    transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=7.0,
                    color=LAYOUT_COLORS[layout])
panel_label(axes_f[0], "f", x=-0.20, y=1.10)
savefig(fig, fig_dir / "fig01_overview.pdf")

In [9]:
def _agg_envelope(df: pd.DataFrame, metric: str, layout: str):
    sub = df[(df.layout == layout) & np.isfinite(df[metric])]
    x = np.array(K_LEVELS)
    mean = np.array([sub[sub.K == k][metric].mean() for k in x])
    lo = np.array([sub[sub.K == k][metric].quantile(0.25) for k in x])
    hi = np.array([sub[sub.K == k][metric].quantile(0.75) for k in x])
    return x, mean, lo, hi

# Figure 2

set_style(); df = pd.read_csv(out_dir / "summary.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.4))
metrics = [("snr_median", "median SNR"), ("effective_rank", r"effective rank"),
           ("nearest_sep", "nearest lead-field separation / noise"), ("resolution_length", "resolution length (mm)")]
for ax, (metric, ylabel), lab in zip(axs.flat, metrics, "abcd"):
    for age in DISPLAY_AGES:
        col = DISPLAY_AGE_COLORS[age]
        for layout in LAYOUTS:
            sub = df[(df.age == age) & (df.layout == layout)].sort_values("K")
            ax.plot(sub.K, sub[metric], color=col, ls=line_style(layout), marker=marker_style(layout), ms=MS, lw=1.15)
    ax.set_xlabel("channels"); ax.set_ylabel(ylabel); ax.grid(True); ax.set_xticks(K_LEVELS); ax.tick_params(axis='x', rotation=35)
    handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], ls="-", lw=3.0, label=f"{a} m") for a in DISPLAY_AGES]
    handles += [Line2D([0], [0], color="#333333", ls="-", lw=1.0, marker="o", ms=MS, label="AA"),
            Line2D([0], [0], color="#333333", ls="--", lw=0.92, marker="s", ms=MS, label="AH")]
    if lab == 'a':
        boxed_legend(ax, handles=handles, loc="upper left", ncol=3, columnspacing=0.55, handlelength=1.88, borderpad=0.5)
    panel_label(ax, lab)

fig.subplots_adjust(bottom=0.19, top=0.97, hspace=0.43, wspace=0.34)
savefig(fig, fig_dir / "fig02_information.pdf")

In [10]:
# Figure 3

set_style(); df = pd.read_csv(out_dir / "summary.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.6))
for ax, metric, ylabel, lab in [(axs[0, 0], "median_error", "median error (mm)", "a"),
                                (axs[0, 1], "p90_error", "90th percentile error (mm)", "b"),
                                (axs[1, 0], "success20", "posterior mass within 20 mm (%)", "c")]:
    for layout in LAYOUTS:
        x, mean, lo, hi = _agg_envelope(df, metric, layout)
        col = LAYOUT_COLORS[layout]
        ax.fill_between(x, lo, hi, color=col, alpha=0.12, linewidth=0)
        ax.plot(x, mean, color=col, ls=line_style(layout), marker=marker_style(layout), ms=MS, label=layout_label(layout))
    ax.set_xlabel("number of channels"); ax.set_ylabel(ylabel); ax.grid(True); ax.set_xticks(K_LEVELS); ax.tick_params(axis='x', rotation=0)
    panel_label(ax, lab)
    ax.set_ylim([0,105])

handles = [Line2D([0], [0], color=LAYOUT_COLORS['AA'], ls="-", lw=1.0, marker="o", ms=MS, label="AA"),
           Line2D([0], [0], color=LAYOUT_COLORS['AH'], ls="--", lw=0.92, marker="s", ms=MS, label="AH")]
boxed_legend(axs[0, 0], handles = handles, loc="upper right", ncol=1, handlelength=1.88)
# boxed_legend(axs[0, 1], handles = handles,  loc="upper right", ncol=1, handlelength=1.88)
# boxed_legend(axs[1, 0], handles = handles,  loc="lower right", ncol=1, handlelength=1.88)

ax = axs[1, 1]
mat = np.zeros((len(AGES), len(K_LEVELS)))
for i, a in enumerate(AGES):
    for j, k in enumerate(K_LEVELS):
        mat[i, j] = df[(df.age == a) & (df.K == k) & (df.layout == "age-adaptive")].median_error.iloc[0]
im = ax.imshow(mat, aspect="auto", origin="lower", cmap="YlGnBu_r")
ax.set_xticks(range(len(K_LEVELS))); ax.set_xticklabels(K_LEVELS, rotation=0)
ax.set_yticks(range(len(AGES))); ax.set_yticklabels(AGES)
ax.set_xlabel("channels"); ax.set_ylabel("age (m)")
cb = plt.colorbar(im, ax=ax, fraction=0.047, pad=0.02); cb.set_label("median error (mm)")
panel_label(ax, "d")
fig.subplots_adjust(hspace=0.44, wspace=0.32)
savefig(fig, fig_dir / "fig03_performance.pdf")

In [18]:
# Figure 4
CHANNEL_NUM_COLORS = {25:RMB["purple"], 41:RMB["blue"], 61:RMB["green"], 113:RMB["red"]}
# [RMB["red"], RMB["blue"], RMB["green"], RMB["brown"], RMB["purple"], RMB["gold"], RMB["gray"]]
DISPLAY_CHANNEL_NUMBERS = [25, 41, 61, 113]

set_style(); df = pd.read_csv(out_dir / "summary.csv"); feas = pd.read_csv(out_dir / "feasibility.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.4))
ax = axs[0, 0]
for layout in LAYOUTS[::-1]:
    for k in DISPLAY_CHANNEL_NUMBERS:
        sub = df[(df.K == k) & (df.layout == layout)].sort_values("age")
        lw = 0.8 if layout == 'adult-helmet' else 1.05
        ax.plot(sub.age, sub.median_error, color=CHANNEL_NUM_COLORS[k], ls=line_style(layout), 
                marker=marker_style(layout), ms=1.5, lw=lw, label=f"{layout_label(layout)}, {k} chs")
ax.set_xlabel("age (m)"); ax.set_ylabel("median error (mm)"); ax.grid(True)
# ax.set_ylim([0,75])
handles = [Line2D([0], [0], color=CHANNEL_NUM_COLORS[k], ls="-", lw=2.0, label=f"{k} chs") for k in DISPLAY_CHANNEL_NUMBERS]
handles += [Line2D([0], [0], color="#333333", ls="-", lw=1.0, marker="o", ms=MS, label="AA"),
        Line2D([0], [0], color="#333333", ls="--", lw=0.92, marker="s", ms=MS, label="AH")]
# boxed_legend(ax, handles=handles, loc="upper right", ncol=3, columnspacing=0.55, handlelength=1.88, borderpad=0.3)
panel_label(ax, "a")

ax = axs[0, 1]
for layout in LAYOUTS[::-1]:
    for k in DISPLAY_CHANNEL_NUMBERS:
        sub = df[(df.K == k) & (df.layout == layout)].sort_values("age")
        lw = 0.8 if layout == 'adult-helmet' else 1.05
        ax.plot(sub.age, sub.snr_median, color=CHANNEL_NUM_COLORS[k], ls=line_style(layout), 
                marker=marker_style(layout), ms=1.5, lw=lw, label=f"{layout_label(layout)}, {k} chs")
ax.set_xlabel("age (m)"); ax.set_ylabel("median SNR"); ax.grid(True); 
boxed_legend(ax, handles=handles, loc="upper right", ncol=3, columnspacing=0.55, handlelength=1.88, borderpad=0.3)
panel_label(ax, "b")

ax = axs[1, 0]
for layout in LAYOUTS[::-1]:
    for k in DISPLAY_CHANNEL_NUMBERS:
        sub = df[(df.K == k) & (df.layout == layout)].sort_values("age")
        lw = 0.8 if layout == 'adult-helmet' else 1.05
        ax.plot(sub.age, sub.effective_rank, color=CHANNEL_NUM_COLORS[k], ls=line_style(layout), 
                marker=marker_style(layout), ms=1.5, lw=lw, label=f"{layout_label(layout)}, {k} chs")
ax.set_xlabel("age (m)"); ax.set_ylabel("effective rank"); ax.grid(True); 
# ax.set_ylim([15,44])
# boxed_legend(ax, handles=handles, loc="lower right", ncol=3, columnspacing=0.55, handlelength=1.88, borderpad=0.3)
panel_label(ax, "c")

ax = axs[1, 1]
sides = [10, 20, 30, 40, 50]
ages = AGES
mat = np.full((len(sides), len(ages)), np.nan)
for i, side in enumerate(sides):
    for j, age in enumerate(ages):
        sub = feas[(feas.layout == "age-adaptive") & (feas.probe_mm == side) & (feas.age == age)]
        if len(sub): mat[i, j] = sub.feasible.iloc[0]
im = ax.imshow(mat, aspect="auto", origin="lower", cmap="YlGnBu")
ax.set_xticks(range(len(ages))); ax.set_xticklabels(ages)
ax.tick_params(axis='x', rotation=45)
ax.set_yticks(range(len(sides))); ax.set_yticklabels(sides)
ax.set_xlabel("age (m)"); ax.set_ylabel("probe base side (mm)")
for i in range(len(sides)):
    for j in range(len(ages)):
        val = mat[i, j]
        ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=6.2, color=("white" if val >= 90 else "#222222"))
cb = plt.colorbar(im, ax=ax, fraction=0.047, pad=0.02); cb.set_label("largest feasible K")
panel_label(ax, "d")
fig.subplots_adjust(hspace=0.42, wspace=0.34)
savefig(fig, fig_dir / "fig04_age_scaling.pdf")

In [12]:
# Figure 5
set_style(); 
noise = pd.read_csv(out_dir / "noise.csv"); 
depth = pd.read_csv(out_dir / "depth.csv"); 
rand = pd.read_csv(out_dir / "random_control.csv"); 
lobe = pd.read_csv(out_dir / "lobe.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.4))
ax = axs[0, 0]
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    for layout in LAYOUTS:
        sub = noise[(noise.layout == layout) & (noise.age == age)].sort_values("noise_ft")
        lw = 0.8 if layout == 'adult-helmet' else 1.05
        ms = 1.2 if layout == 'adult-helmet' else 1.5
        ax.plot(sub.noise_ft, sub.median_error, marker=marker_style(layout), 
                ms=ms, color=col, ls=line_style(layout), lw=lw, label=f"{age} m, {layout_label(layout)}")
ax.set_xlabel("sensor noise (fT)"); ax.set_ylabel("median error (mm)"); ax.grid(True)
# Compact legend using age color and layout style, placed on the left side to avoid high-noise curves.
handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=2, label=f"{a} M") for a in DISPLAY_AGES]
handles += [Line2D([0], [0], color="#333333", ls="-", lw=0.8, marker='o', ms=1.5, label="AA"),
            Line2D([0], [0], color="#333333", ls="--", lw=0.75, marker='s', ms=1.2, label="AH")]
# ax.set_ylim([0,100])
boxed_legend(ax, handles=handles, loc="upper left", ncol=2, handlelength=1.5, borderpad=0.22)
panel_label(ax, "a")

ax = axs[0, 1]
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    for layout in LAYOUTS:
        sub = depth[(depth.layout == layout) & (depth.age == age)].sort_values("depth_mm")
        lw = 0.8 if layout == 'adult-helmet' else 1.05
        ms = 1.2 if layout == 'adult-helmet' else 1.5
        ax.plot(sub.depth_mm, sub.median_error, marker=marker_style(layout), 
                ms=ms, color=col, ls=line_style(layout), lw=lw)
ax.set_xlabel("source depth (mm)"); ax.set_ylabel("median error (mm)"); ax.grid(True)
# ax.set_ylim([0,100])
# boxed_legend(ax, handles=handles, loc="center left", ncol=1, handlelength=1.1, borderpad=0.22)
panel_label(ax, "b")

ax = axs[1, 0]
summ = pd.read_csv(out_dir / "summary.csv")
ages = DISPLAY_AGES
rand_stats = rand.groupby("age").median_error.quantile([0.10, 0.50, 0.90]).unstack()
sym = summ[(summ.layout == "age-adaptive") & (summ.K == 61) & (summ.age.isin(ages))].sort_values("age")
ax.fill_between(ages, rand_stats.loc[ages, 0.10], rand_stats.loc[ages, 0.90], color="#C9D3DD", alpha=0.55, label="random phase 10-90%")
ax.plot(ages, rand_stats.loc[ages, 0.50], color="#6B7280", ls="--", marker="s", ms=MS, label="random median")
ax.plot(sym.age, sym.median_error, color=LAYOUT_COLORS["age-adaptive"], ls="-", marker="o", ms=MS, label="symmetric template")
ax.set_xlabel("age (m)"); ax.set_ylabel("median error (mm)")
ax.grid(True); boxed_legend(ax, loc="upper left", handlelength=1.2)
panel_label(ax, "c")

ax = axs[1, 1]
avg = lobe.groupby(["layout", "lobe"]).median_error.mean().reset_index()
x = np.arange(len(LOBES)); w = 0.32
for i, layout in enumerate(LAYOUTS[::-1]):
    vals = [avg[(avg.layout == layout) & (avg.lobe == lb)].median_error.iloc[0] for lb in LOBES]
    ax.barh(x + (i - 0.5) * w, vals, height=w, color=LAYOUT_COLORS[layout], alpha=0.78, label=layout_label(layout))
ax.set_yticks(x); ax.set_yticklabels([SHORT_LOBE[l] for l in LOBES])
ax.set_xlabel("median error (mm)"); ax.grid(True, axis="x"); 
# boxed_legend(ax, loc="lower right")
panel_label(ax, "d")
fig.subplots_adjust(hspace=0.42, wspace=0.34)
savefig(fig, fig_dir / "fig05_robustness.pdf")

In [13]:
def source_expected_error(age: int, layout: str, k: int) -> np.ndarray:
    L = leadfield(age, layout, k)
    D = source_distance_matrix_mm(age)
    G = L.T @ L; n2 = np.diag(G)
    delta = np.maximum(n2[:, None] + n2[None, :] - 2 * G, 0) / (2 * NOISE_FT * NOISE_FT)
    logits = -delta; logits -= np.max(logits, axis=1, keepdims=True)
    P = np.exp(logits); P /= np.sum(P, axis=1, keepdims=True)
    return np.sum(P * D, axis=1)

# Figure 6
set_style(); lobe = pd.read_csv(out_dir / "lobe.csv"); summ = pd.read_csv(out_dir / "summary.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.2))
ax = axs[0, 0]
avg = lobe.groupby(["layout", "lobe"]).success20.mean().reset_index()
x = np.arange(len(LOBES)); w = 0.32
for i, layout in enumerate(LAYOUTS):
    vals = [avg[(avg.layout == layout) & (avg.lobe == lb)].success20.iloc[0] for lb in LOBES]
    ax.bar(x + (i - 0.5) * w, vals, width=w, color=LAYOUT_COLORS[layout], alpha=0.78, label=layout_label(layout))
ax.set_xticks(x); ax.set_xticklabels([SHORT_LOBE[l] for l in LOBES], rotation=20)
ax.set_ylabel("posterior mass within 20 mm (%)"); ax.grid(True, axis="y"); boxed_legend(ax, loc="upper right")
ax.set_ylim([0,105])
panel_label(ax, "a")

ax = axs[0, 1]
for layout in LAYOUTS:
    sub = summ[(summ.layout == layout) & (summ.K == 61)].sort_values("age")
    ax.plot(sub.age, sub.success20, color=LAYOUT_COLORS[layout], ls=line_style(layout), marker=marker_style(layout), ms=MS, label=layout_label(layout))
ax.set_xlabel("age (m)"); ax.set_ylabel("posterior mass within 20 mm (%)"); ax.grid(True); boxed_legend(ax, loc="upper right")
ax.set_ylim([0,105])
panel_label(ax, "b")

ax = axs[1, 0]
for layout in LAYOUTS:
    sub = summ[(summ.layout == layout) & (summ.K == 61)].sort_values("age")
    ax.plot(sub.age, sub.median_r50, color=LAYOUT_COLORS[layout], ls=":", 
            marker=marker_style(layout), ms=MS, label=f"{layout_label(layout)}: 50%")
    ax.plot(sub.age, sub.median_r90, color=LAYOUT_COLORS[layout], 
            ls="-", marker=marker_style(layout), ms=MS, label=f"{layout_label(layout)}: 90%")
ax.set_xlabel("age (m)"); ax.set_ylabel("median credible radius (mm)"); ax.grid(True); 
boxed_legend(ax, loc="upper right", ncol=1, handlelength=2.2)
panel_label(ax, "c")

ax = axs[1, 1]
xy = np.asarray(source_grid()[1])
err_a = source_expected_error(120, "age-adaptive", 61)
err_h = source_expected_error(120, "adult-helmet", 61)
improvement = np.clip(err_h - err_a, 0, None)
cap_scaffold(ax, radius=SOURCE_LAMBERT_RADIUS, midline=True)
sc = ax.scatter(xy[:, 0], xy[:, 1], c=improvement, s=SCATTER_S, cmap="YlGnBu", edgecolors="white", linewidths=0.18)
cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02); cb.set_label("adult-helmet minus age-adaptive error (mm)")
panel_label(ax, "d", x=-0.12)
fig.subplots_adjust(hspace=0.42, wspace=0.42)
savefig(fig, fig_dir / "fig06_source_space.pdf")

In [20]:
# Figure 7
set_style(); df = pd.read_csv(out_dir / "summary.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.2))
ax = axs[0, 0]
for age in DISPLAY_AGES:
    for layout in LAYOUTS:
        L = leadfield(age, layout, 61) / NOISE_FT
        gram = L @ L.T
        evals = np.linalg.eigvalsh(0.5 * (gram + gram.T))
        s = np.sqrt(np.maximum(evals, 1e-18))[::-1]
        mass = np.cumsum(s) / np.sum(s) * 100.0
        ax.plot(np.arange(1, len(mass) + 1), mass, color=DISPLAY_AGE_COLORS[age], 
                ls=line_style(layout), marker=marker_style(layout), markevery=8, ms=MS)
ax.set_xlabel("singular modes retained"); ax.set_ylabel("cumulative singular-value mass (%)"); ax.grid(True)
handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=2, label=f"{a} m") for a in DISPLAY_AGES]
handles += [Line2D([0], [0], color="#333333", ls="-", lw=0.8, marker='o', ms=1.5, label="AA"),
            Line2D([0], [0], color="#333333", ls="--", lw=0.75, marker='s', ms=1.2, label="AH")]
boxed_legend(ax, handles=handles, loc="lower right", ncol=2, handlelength=1.5, borderpad=0.22)
panel_label(ax, "a")

ax = axs[0, 1]
for layout in LAYOUTS:
    sub = df[df.layout == layout].groupby("K").effective_rank.mean()
    ax.plot(sub.index, sub.values, color=LAYOUT_COLORS[layout], ls=line_style(layout), marker=marker_style(layout), ms=MS, label=layout_label(layout))
ax.set_xlabel("channels"); ax.set_ylabel("mean effective rank"); ax.set_xticks(K_LEVELS); ax.tick_params(axis='x', rotation=0)
ax.grid(True); boxed_legend(ax, loc="lower right")
panel_label(ax, "b")

ax = axs[1, 0]
for layout in LAYOUTS:
    sub = df[df.layout == layout].groupby("K").agg({"effective_rank":"mean", "nearest_sep":"mean"}).reset_index()
    ax.plot(sub.effective_rank, sub.nearest_sep, color=LAYOUT_COLORS[layout], ls=line_style(layout), marker=marker_style(layout), 
            ms=MS, label=layout_label(layout))
    for _, row in sub.iterrows():
        if int(row.K) in [5, 41, 113]:
            ax.text(row.effective_rank, row.nearest_sep, f" {int(row.K)}", fontsize=6, va="center", color=LAYOUT_COLORS[layout])
ax.set_xlabel("effective rank"); ax.set_ylabel("nearest separation / noise"); ax.grid(True); boxed_legend(ax, loc="lower right")
panel_label(ax, "c")

ax = axs[1, 1]
bins = np.linspace(0, 120, 9)
for age in DISPLAY_AGES:
    D = source_distance_matrix_mm(age)
    upper = np.triu_indices_from(D, k=1)
    for layout in LAYOUTS:
        L = leadfield(age, layout, 61) / NOISE_FT
        G = L.T @ L; n2 = np.diag(G); sep = np.sqrt(np.maximum(n2[:, None] + n2[None, :] - 2 * G, 0))[upper]
        dist = D[upper]
        bx, by = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            m = (dist >= lo) & (dist < hi)
            if np.sum(m) > 8:
                bx.append((lo + hi) / 2); by.append(np.median(sep[m]))
        by = list(np.maximum.accumulate(np.asarray(by)))
        ax.plot(bx, by, color=DISPLAY_AGE_COLORS[age], ls=line_style(layout), marker=marker_style(layout), ms=MS, label=layout_label(layout))
ax.set_xlabel("source separation (mm)"); ax.set_ylabel("lead-field separation / noise"); ax.grid(True); 
handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=2, label=f"{a} m") for a in DISPLAY_AGES]
handles += [Line2D([0], [0], color="#333333", ls="-", lw=0.8, marker='o', ms=1.5, label="AA"),
            Line2D([0], [0], color="#333333", ls="--", lw=0.75, marker='s', ms=1.2, label="AH")]
boxed_legend(ax, handles=handles, loc="lower right", ncol=2, handlelength=1.5, borderpad=0.22)
panel_label(ax, "d")
fig.subplots_adjust(hspace=0.40, wspace=0.34)
savefig(fig, fig_dir / "fig07_information_geometry.pdf")

In [19]:
def smallest_compatible_probe(cell: int) -> int:
    probes = [p for p in PROBE_SIDES_MM if p >= cell]
    return min(probes) if probes else max(PROBE_SIDES_MM)


def aperture_metrics(age: int, layout: str, k: int, cell: int) -> dict:
    # Aperture-only metric: row positions are held fixed to isolate cell averaging from mechanical packing.
    return posterior_metrics(leadfield(age, layout, k, cell), age)


chs = 41
set_style(); feas = pd.read_csv(out_dir / "feasibility.csv")
fig, axs = plt.subplots(2, 2, figsize=cm_to_inch(18, 13.2))
cells = CELL_DIAMETERS_MM
handles = [Line2D([0], [0], color=DISPLAY_AGE_COLORS[a], lw=2, label=f"{a} m") for a in DISPLAY_AGES]
handles += [Line2D([0], [0], color="#333333", ls="-", lw=0.8, marker='o', ms=1.5, label="AA"),
            Line2D([0], [0], color="#333333", ls="--", lw=0.75, marker='s', ms=1.2, label="AH")]

ax = axs[0, 0]
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    for layout in LAYOUTS:
        ys = [aperture_metrics(age, layout, chs, int(cell))["median_error"] for cell in cells]
        ax.plot(cells, ys, color=col, ls=line_style(layout), marker=marker_style(layout), ms=MS)
ax.set_xlabel("atomic-cell effective diameter (mm)"); ax.set_ylabel(f"median error @ {chs} chs (mm)"); ax.grid(True)
boxed_legend(ax, handles=handles, loc="lower right", ncol=2, handlelength=1.5, borderpad=0.22)
panel_label(ax, "a")

ax = axs[0, 1]
for age in DISPLAY_AGES:
    col = DISPLAY_AGE_COLORS[age]
    for layout in LAYOUTS:
        ys = [aperture_metrics(age, layout, chs, int(cell))["effective_rank"] for cell in cells]
        ax.plot(cells, ys, color=col, ls=line_style(layout), marker=marker_style(layout), ms=MS)
ax.set_xlabel("atomic-cell effective diameter (mm)"); ax.set_ylabel(f"effective rank @ {chs} chs"); ax.grid(True)
# boxed_legend(ax, handles=handles, loc="lower right", ncol=1, handlelength=1.1, borderpad=0.22)
panel_label(ax, "b")

ax = axs[1, 0]
ages = AGES
cell_rows = [5, 10, 15, 20, 25]
mat = np.full((len(cell_rows), len(ages)), np.nan)
for i, cell in enumerate(cell_rows):
    for j, age in enumerate(ages):
        base = aperture_metrics(age, "age-adaptive", 41, 0)["median_error"]
        val = aperture_metrics(age, "age-adaptive", 41, int(cell))["median_error"]
        mat[i, j] = val - base
im = ax.imshow(mat, aspect="auto", origin="lower", cmap="YlGnBu")
ax.set_xticks(range(len(ages))); ax.set_xticklabels(ages)
ax.set_yticks(range(len(cell_rows))); ax.set_yticklabels(cell_rows)
ax.set_xlabel("age (m)"); ax.set_ylabel("cell diameter (mm)")
ax.tick_params(axis='x', rotation=45)
vmax = np.nanmax(mat)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat[i, j]
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=5.8, color=("white" if val > 0.65 * vmax else "#222222"))
cb = plt.colorbar(im, ax=ax, fraction=0.047, pad=0.02); cb.set_label("aperture-only error penalty (mm)")
panel_label(ax, "c")

ax = axs[1, 1]
sides = [10, 20, 30, 40, 50]
mat2 = np.full((len(sides), len(ages)), np.nan)
for i, side in enumerate(sides):
    for j, age in enumerate(ages):
        sub = feas[(feas.layout == "age-adaptive") & (feas.probe_mm == side) & (feas.age == age)]
        if len(sub): mat2[i, j] = sub.feasible.iloc[0]
im2 = ax.imshow(mat2, aspect="auto", origin="lower", cmap="YlGnBu")
ax.set_xticks(range(len(ages))); ax.set_xticklabels(ages)
ax.set_yticks(range(len(sides))); ax.set_yticklabels(sides)
ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("age (m)"); ax.set_ylabel("probe base side (mm)")
for i in range(mat2.shape[0]):
    for j in range(mat2.shape[1]):
        val = mat2[i, j]
        ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=5.8, color=("white" if val >= 90 else "#222222"))
cb2 = plt.colorbar(im2, ax=ax, fraction=0.047, pad=0.02); cb2.set_label("largest feasible K")
panel_label(ax, "d")
fig.subplots_adjust(hspace=0.40, wspace=0.34)
savefig(fig, fig_dir / "fig08_finite_footprint.pdf")